# Playground — Step 2b: True Beam-Search Guided Generation

The original `generate_with_topk_guide` keeps the k children of a **single winning parent**
at each step — its beams only ever differ in the last token (greedy with k-lookahead).

The new `generate_with_topk_beam_guide` lets **all pool candidates compete**: the top-k
sequences survive regardless of parent, so beams genuinely diverge. Cost: a k²-wide
forward pass per step after the first (mind memory for large `k` × `batch_size`).

Sections:
1. Original vs beam, same concept and settings
2. Probe trajectory comparison
3. k=1 degeneracy check (both must collapse to identical greedy decoding)
4. Multi-concept beam steering

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import matplotlib.pyplot as plt
import torch

from frames.representations import FrameUnembeddingRepresentation

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 3
STEPS = 16


def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


PROMPTS = [
    chat("Tell me a short story.", "Once"),
    chat("Name things that make people happy.", "1."),
]

## 1 — Original (greedy + k-lookahead) vs true beam search

Same concept, same `k`, same `steps`. The beam variant explores a strictly larger
search space, so its final cumulative concept score should be ≥ the original's
(occasional exceptions possible since both score candidates one step delayed).

In [ ]:
GUIDE = ["joy.n.01"]  # try ["dog.n.01"], or a differential like ["woman.n.01", "man.n.01"]

texts_orig, probe_orig = fur.quick_generate_with_topk_guide(
    PROMPTS,
    guide=GUIDE,
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)
texts_beam, probe_beam = fur.quick_generate_with_topk_beam_guide(
    PROMPTS,
    guides=[GUIDE],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)

for orig, beam in zip(texts_orig, texts_beam):
    print("ORIG:", answer(orig)[:160])
    print("BEAM:", answer(beam)[:160])
    print()

## 2 — Probe trajectories

Cumulative concept-alignment score over token positions, original vs beam.

In [ ]:
fig, axes = plt.subplots(1, len(PROMPTS), figsize=(12, 3.5), sharey=True)
for i, ax in enumerate(axes):
    ax.plot(probe_orig[i].float(), label="original")
    ax.plot(probe_beam[i].float(), label="beam")
    ax.set_title(f"prompt {i}")
    ax.set_xlabel("token position")
axes[0].set_ylabel("cumulative score")
axes[0].legend()
plt.tight_layout()
plt.show()

print("final scores:")
print("  original:", probe_orig[:, -1].tolist())
print("  beam:    ", probe_beam[:, -1].tolist())

## 3 — k=1 degeneracy check

With `k=1` there is nothing to select in either method — both must produce plain
greedy decoding, token for token, regardless of the concept.

In [ ]:
texts_orig_k1, _ = fur.quick_generate_with_topk_guide(
    PROMPTS,
    guide=GUIDE,
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=1,
    steps=STEPS,
)
texts_beam_k1, _ = fur.quick_generate_with_topk_beam_guide(
    PROMPTS,
    guides=[GUIDE],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=1,
    steps=STEPS,
)

assert texts_orig_k1 == texts_beam_k1, "k=1 outputs must be identical"
print("k=1 degeneracy check passed — identical greedy outputs:")
for t in texts_beam_k1:
    print(" ", answer(t)[:120])

## 4 — Multi-concept beam steering

The beam method accepts the same concepts/weights/scorer as the unified loop,
so all multi-concept modes work under beam decoding too.

In [ ]:
texts_multi, probe_multi = fur.quick_generate_with_topk_beam_guide(
    PROMPTS,
    guides=[["joy.n.01"], ["dog.n.01"]],
    weights=[0.6, 0.4],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)
for t in texts_multi:
    print(answer(t)[:160], "\n")